# QGym training on Colab

**Runtime → Change runtime type → GPU** before running anything.

| Cell | Does |
|---|---|
| 1 | Mounts Drive, installs `uv`, clones `jt/sdk`, builds the venv. Re-run after every reconnect. |
| 2 | Trains. Set task / iterations / envs at the top. |
| 3 | Copies the newest checkpoint (+ its saved configs) to Drive. |

Colab wipes the VM on disconnect or runtime-type change, so cell 1 is written to be
idempotent — re-running it on a live session is a no-op that takes seconds.
Only Drive survives: checkpoints go there via cell 3, and warp's JIT kernel cache is
kept there too so the multi-minute first-run compile happens only once, ever.

`unitree-sdk2py` and `cyclonedds` are deliberately **not** installed — they are only
imported by `go2_deploy/` (Go2 hardware deployment) and are not needed to train.

In [ ]:
# ============================== CELL 1: SETUP ==============================
BRANCH = "jt/sdk"
REPO_URL = "https://github.com/LampLighterLab/QGym.git"
REPO = "/content/QGym"
DRIVE = "/content/drive/MyDrive/QGym"

import os, subprocess, textwrap

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE}/warp_cache", exist_ok=True)

# uv installs to ~/.local/bin; warp caches compiled kernels on Drive.
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
os.environ["WARP_CACHE_PATH"] = f"{DRIVE}/warp_cache"
# Colab exports MPLBACKEND=module://matplotlib_inline.backend_inline, but
# matplotlib_inline lives in Colab's system 3.12 -- not this 3.11 venv -- so
# the venv's matplotlib rejects that backend name on import. Force headless.
os.environ["MPLBACKEND"] = "Agg"

setup = textwrap.dedent(f"""
    set -euo pipefail

    command -v uv >/dev/null || curl -LsSf https://astral.sh/uv/install.sh | sh

    if [ -d {REPO}/.git ]; then
        git -C {REPO} fetch origin {BRANCH}
        git -C {REPO} checkout {BRANCH}
        git -C {REPO} pull --ff-only
    else
        git clone --branch {BRANCH} {REPO_URL} {REPO}
    fi

    cd {REPO}
    uv python install 3.11          # pyproject pins requires-python == 3.11.*
    # --extra gpu pulls mujoco-warp (CUDA); --no-dev skips pytest/ruff/marimo.
    uv sync --frozen --extra gpu --no-dev || uv sync --extra gpu --no-dev
""")
subprocess.run(["bash", "-c", setup], check=True)

check = "import torch, mujoco, mujoco_warp; " \
        "print('torch', torch.__version__, '| mujoco', mujoco.__version__, " \
        "'| cuda', torch.cuda.is_available(), " \
        "'|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')"
subprocess.run(
    ["uv", "run", "--frozen", "--extra", "gpu", "--no-dev", "python", "-c", check],
    cwd=REPO, check=True,
)
print("\nReady. If cuda is False, set Runtime -> Change runtime type -> GPU and re-run.")

In [ ]:
# ============================= CELL 2: TRAIN ==============================
TASK            = "go2trot"   # go2trot | mini_cheetah | humanoid | pendulum | ...
MAX_ITERATIONS  = 300
NUM_ENVS        = 4096
DEVICE          = "cuda:0"
EXPERIMENT_NAME = None        # None -> the task's default logs/<experiment_name>/
RESUME          = False       # True -> continue the newest run for this experiment
USE_WANDB       = False       # True requires WANDB_API_KEY in the env (else it hangs)

REPO = "/content/QGym"
DRIVE = "/content/drive/MyDrive/QGym"

import os, subprocess, sys
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
os.environ["WARP_CACHE_PATH"] = f"{DRIVE}/warp_cache"
# Colab exports MPLBACKEND=module://matplotlib_inline.backend_inline, but
# matplotlib_inline lives in Colab's system 3.12 -- not this 3.11 venv -- so
# the venv's matplotlib rejects that backend name on import. Force headless.
os.environ["MPLBACKEND"] = "Agg"

cmd = [
    "uv", "run", "--frozen", "--extra", "gpu", "--no-dev", "scripts/train.py",
    "--task", TASK,
    "--device", DEVICE,
    "--num_envs", str(NUM_ENVS),
    "--max_iterations", str(MAX_ITERATIONS),
    "--headless",                     # no display on Colab
]
if EXPERIMENT_NAME:
    cmd += ["--experiment_name", EXPERIMENT_NAME]
if RESUME:
    cmd += ["--resume"]
if not USE_WANDB:
    cmd += ["--disable_wandb"]

print(" ".join(cmd), "\n", flush=True)
# First warp run JIT-compiles kernels (minutes, cached on Drive afterwards).
proc = subprocess.Popen(
    cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    sys.exit(f"training exited with code {proc.returncode}")

In [ ]:
# ==================== CELL 3: SAVE CHECKPOINTS TO DRIVE ====================
REPO = "/content/QGym"
DRIVE = "/content/drive/MyDrive/QGym"
ALL_RUNS = False   # False -> newest run only; True -> every run under logs/

import re, shutil
from pathlib import Path

def iteration(p):
    return int(re.search(r"model_(\d+)\.pt", p.name).group(1))

logs = Path(REPO) / "logs"
# Layout: logs/<experiment_name>/<Mon##_HH-MM-SS>_<run_name>/model_<iter>.pt
runs = sorted(
    {p.parent for p in logs.glob("*/*/model_*.pt")},
    key=lambda d: max(p.stat().st_mtime for p in d.glob("model_*.pt")),
)
if not runs:
    raise SystemExit(f"no checkpoints found under {logs}")
if not ALL_RUNS:
    runs = runs[-1:]

total_copied = total_bytes = total_skipped = 0
for run in runs:
    dest = Path(DRIVE) / "checkpoints" / run.parent.name / run.name
    dest.mkdir(parents=True, exist_ok=True)

    copied = []
    for ckpt in sorted(run.glob("model_*.pt"), key=iteration):
        target = dest / ckpt.name
        # Size match == already synced. Makes re-runs mid-training incremental,
        # so you can call this cell repeatedly without recopying gigabytes.
        if target.exists() and target.stat().st_size == ckpt.stat().st_size:
            total_skipped += 1
            continue
        shutil.copy2(ckpt, target)
        copied.append(ckpt)

    # The run's saved configs travel with it -- resume/play need them.
    for extra in run.iterdir():
        if extra.is_file() and not extra.name.startswith("model_"):
            shutil.copy2(extra, dest / extra.name)
        elif extra.is_dir():
            shutil.copytree(extra, dest / extra.name, dirs_exist_ok=True)

    nbytes = sum(c.stat().st_size for c in copied)
    total_copied += len(copied)
    total_bytes += nbytes
    print(f"{run.parent.name}/{run.name}: +{len(copied)} ckpt ({nbytes / 1e6:.1f} MB)")
    if copied:
        print(f"    iters {iteration(copied[0])}..{iteration(copied[-1])} -> {dest}")

print(f"\ncopied {total_copied} checkpoint(s), {total_bytes / 1e6:.1f} MB; "
      f"{total_skipped} already on Drive")